<a href="https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abiodunadesesan/FlyRank-ml-Internship-/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

### Research Question

Which visible pages have the greatest opportunity for CTR improvement, and how can we rank them for content optimization?

### Decision

Which webpages should be prioritized for CTR optimization?

### Who will use the result?

Content and SEO teams can use the ranking to decide which pages to review first for possible improvements to titles, meta descriptions, content, and other on-page elements.

### Objective

Build a repeatable CTR opportunity scoring approach using the FlyRank search dataset, compare it with a transparent baseline, validate the result honestly, and turn the analysis into a ranked action playbook.

In [ ]:
import duckdb
import pandas as pd

print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)

DuckDB: 1.3.2
Pandas: 2.2.3


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("FlyRank warehouse authentication is ready.")

FlyRank warehouse authentication is ready.


In [ ]:
# Inspect the available FlyRank warehouse tables

tables = con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'main'
ORDER BY table_name
""").df()

tables

,table_name


In [ ]:
# Check whether the Hugging Face secret is available
print("Token available:", bool(HF_TOKEN))

Token available: True


In [ ]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

print("FlyRank warehouse authentication is ready.")

FlyRank warehouse authentication is ready.


In [ ]:
rel = "hf://datasets/FlyRank/internship-warehouse"

test = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

test

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,78835655


In [ ]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

schema

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [ ]:
summary = con.sql(f"""
SELECT
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    COUNT(DISTINCT CONCAT(
        CAST(report_date AS VARCHAR), '|',
        client_hash_id, '|',
        content_hash_id
    )) AS unique_grain_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,start_date,end_date,total_rows,clients,contents,unique_grain_rows
0,2025-01-27,2026-06-30,78835655,70,427292,78829265


In [ ]:
duplicates = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 20
""").df()

duplicates

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-17,client_1a730cb2640a1abf,content_965b9031838c130f,2
1,2026-06-17,client_1a730cb2640a1abf,content_5c5b96cd4d1829d8,2
2,2026-06-18,client_def0955f7a377868,content_40c868cfcb49537c,2
3,2026-06-20,client_1a730cb2640a1abf,content_6af788521680129d,2
4,2026-06-19,client_06d356715a8ff3b6,content_6c6a2658f025ec02,2
5,2026-06-21,client_1a8bf67cad4ee525,content_567e7aa5ce2a36e6,2
6,2026-06-22,client_8ddc46da5414ffd8,content_55335cfcf3499724,2
7,2026-06-23,client_1a8bf67cad4ee525,content_e85eff1aad797f4c,2
8,2026-06-23,client_1a8bf67cad4ee525,content_2630830d5f397c6c,2
9,2026-06-23,client_1a8bf67cad4ee525,content_fb1b290d16ca0d29,2


In [4]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("FlyRank warehouse connection restored.")

FlyRank warehouse connection restored.


In [7]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)

Pandas: 2.2.3


In [10]:
import pandas as pd
import numpy as np
import duckdb
from google.colab import userdata

# Restore authentication
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN was not found in Colab Secrets.")

# Restore DuckDB connection
con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# Restore warehouse location
rel = "hf://datasets/FlyRank/internship-warehouse"

# Restore time windows
TRAIN_START = "2025-10-01"
TRAIN_END   = "2026-03-31"

VALID_START = "2026-04-01"
VALID_END   = "2026-06-30"

print("Capstone state restored.")
print("Training:", TRAIN_START, "to", TRAIN_END)
print("Validation:", VALID_START, "to", VALID_END)

Capstone state restored.
Training: 2025-10-01 to 2026-03-31
Validation: 2026-04-01 to 2026-06-30


In [11]:
page_features = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
),

page_features AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,

        SUM(gsc_impressions * gsc_avg_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS avg_position,

        COUNT(DISTINCT report_date) AS active_days

    FROM clean_data

    WHERE report_date BETWEEN '{TRAIN_START}' AND '{TRAIN_END}'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM page_features
WHERE impressions >= 100
""").df()

print("Training pages:", len(page_features))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Training pages: 126622


In [12]:
validation_outcomes = con.sql(f"""
WITH clean_data AS (
    SELECT DISTINCT *
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    WHERE gsc_data_available = TRUE
      AND gsc_impressions > 0
),

future_pages AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS validation_impressions,
        SUM(gsc_clicks) AS validation_clicks,

        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS validation_ctr,

        SUM(gsc_impressions * gsc_avg_position) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS validation_avg_position,

        COUNT(DISTINCT report_date) AS validation_active_days

    FROM clean_data

    WHERE report_date BETWEEN '{VALID_START}' AND '{VALID_END}'

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT *
FROM future_pages
WHERE validation_impressions >= 100
""").df()

print("Validation pages:", len(validation_outcomes))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Validation pages: 151996


In [13]:
model_dataset = page_features.merge(
    validation_outcomes,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

print("Pages in both periods:", len(model_dataset))

Pages in both periods: 104173


In [14]:
model_dataset["position_bucket"] = pd.cut(
    model_dataset["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "4-5", "6-10", "11-20", "21+"],
    right=True
)

print(model_dataset["position_bucket"].value_counts().sort_index())

position_bucket
1-3      10519
4-5      17472
6-10     33221
11-20    21735
21+      21226
Name: count, dtype: int64


In [15]:
baseline_ctr = {
    "1-3": 0.005443,
    "4-5": 0.004597,
    "6-10": 0.003267,
    "11-20": 0.003472,
    "21+": 0.001559
}

model_dataset["baseline_ctr"] = (
    model_dataset["position_bucket"]
    .astype(str)
    .map(baseline_ctr)
)

print(
    "Pages with baseline CTR:",
    model_dataset["baseline_ctr"].notna().sum()
)

model_dataset[
    ["position_bucket", "ctr", "baseline_ctr"]
].head(10)

Pages with baseline CTR: 104173


,position_bucket,ctr,baseline_ctr
0,21+,0.001198,0.001559
1,21+,0.005769,0.001559
2,21+,0.007663,0.001559
3,21+,0.002451,0.001559
4,11-20,0.005325,0.003472
5,4-5,0.001078,0.004597
6,21+,0.002463,0.001559
7,6-10,0.000663,0.003267
8,21+,0.009412,0.001559
9,21+,0.003914,0.001559


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
